# Breast Ultrasound Classification — Colab Training

Runs the **same scripts** as local execution, on a free GPU.

Training ResNet50 / EfficientNet-B0 at 224x224 is slow on CPU; this notebook is
the practical way to run the architecture comparison.

**Before you start:** Runtime → Change runtime type → Hardware accelerator → **GPU**

At the end you download `models/` and `results/` and drop them into your local
project. The Flask app and the research dashboard read them directly.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Get the project

Either clone your repository, or upload a zip of the project folder
(excluding `.venv/`, `dataset/` and `data/`).


In [ ]:
# Option A -- clone (replace with your repository URL)
# !git clone https://github.com/<you>/<repo>.git project

# Option B -- upload a zip of the project
from google.colab import files
import zipfile, os

uploaded = files.upload()                      # choose your project .zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as archive:
    archive.extractall("project")

# Descend into the folder that actually contains app.py
root = "project"
for current, dirs, filenames in os.walk("project"):
    if "app.py" in filenames and "src" in dirs:
        root = current
        break
os.chdir(root)
print("Working directory:", os.getcwd())
!ls

## 3. Install dependencies

Colab already ships a recent TensorFlow. Installing the pinned 2.15 would force
a long reinstall and a runtime restart, so only the missing packages are added
here. The training and evaluation code does not depend on TF 2.15 specifically —
that pin exists locally so the legacy `.h5` model still loads.


In [ ]:
!pip install -q opencv-python-headless scikit-learn matplotlib seaborn pytest
import tensorflow as tf
print("Using TensorFlow", tf.__version__)

## 4. Get the BUSI dataset

BUSI (Al-Dhabyani et al. 2020): 780 images — 437 benign, 210 malignant, 133 normal.

Upload the dataset zip, or mount Drive if you keep it there.


In [ ]:
# Option A -- upload the BUSI zip
from google.colab import files
import zipfile

uploaded = files.upload()                      # Dataset_BUSI_with_GT.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as archive:
    archive.extractall("/content/busi_raw")

# Option B -- from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

!find /content/busi_raw -maxdepth 3 -type d

In [ ]:
# Point --source at the folder containing benign/ malignant/ normal/
!python scripts/prepare_data.py --source /content/busi_raw/Dataset_BUSI_with_GT --force

## 5. Dataset analysis and the shared split

This writes `results/split.json`. **Every model trains on this same split** —
that is what makes the architecture comparison fair.


In [ ]:
!python scripts/analyze_dataset.py

## 6. Train

Each run saves `models/<name>.keras` plus metadata and history.
The test split is never passed to `fit`.


In [ ]:
!python scripts/train.py --config custom_cnn

In [ ]:
!python scripts/train.py --config resnet50

In [ ]:
!python scripts/train.py --config efficientnetb0

### Ablation arms (optional)

These answer specific research questions rather than adding models for their own sake.


In [ ]:
# Does augmentation help?
!python scripts/train.py --config custom_cnn --no-augmentation

# Does input resolution matter? (64x64, the thesis's original setting)
!python scripts/train.py --config custom_cnn_legacy64

## 7. Evaluate on the held-out test split

This is the only step that touches the test set.


In [ ]:
!python scripts/evaluate.py --model custom_cnn
!python scripts/evaluate.py --model resnet50
!python scripts/evaluate.py --model efficientnetb0

In [ ]:
# Ablation arms, if trained above
!python scripts/evaluate.py --model custom_cnn_noaug
!python scripts/evaluate.py --model custom_cnn_legacy64

## 8. Model comparison

Prints the thesis comparison table and flags any model whose confidence interval
overlaps the leader — i.e. where the difference is not statistically distinguishable.


In [ ]:
!python scripts/compare_models.py

## 9. Inspect the figures

In [ ]:
from IPython.display import Image, display
import glob, os

for path in sorted(glob.glob("results/figures/*.png")):
    print(os.path.basename(path))
    display(Image(path, width=520))

## 10. Download the results

Unzip these into your local project root. The Flask app picks up `models/`
automatically and `/research` renders `results/`.


In [ ]:
!zip -qr trained_artifacts.zip models results
from google.colab import files
files.download("trained_artifacts.zip")

---

### Reminder

Numbers produced here are the real, leakage-free results. If they are lower than
the 99.6% previously reported, that is the point: the earlier figure came from a
test set in which 77.3% of images had a near-duplicate in training.

An honest 85% is a defensible result. A leaked 99.6% is not.
